# 05 K-Means Clustering Fraud Analysis Model

## Model Plan


In [ ]:
from pathlib import Path
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import MiniBatchKMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_auc_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

PROJECT_ROOT = Path("..").resolve()
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SIZE = 500_000
N_CLUSTERS = 2

if not DATA_FILE.exists():
    raise FileNotFoundError("Run src/01_process_data.py first to create cleaned_transactions.csv")

## Load Cleaned Data

In [ ]:
use_columns = [
    "id", "date", "amount", "use_chip", "merchant_state", "mcc", "errors", "is_fraud"
]

transactions = pd.read_csv(DATA_FILE, usecols=use_columns)
transactions.shape

In [ ]:
transactions["is_fraud"].value_counts(normalize=True).mul(100).round(4).rename("percent")

## Feature Engineering


In [ ]:
model_data = transactions.copy()

model_data["date"] = pd.to_datetime(model_data["date"], errors="coerce")
model_data["amount_value"] = pd.to_numeric(
    model_data["amount"].astype(str).str.replace("$", "", regex=False),
    errors="coerce",
)
model_data["abs_amount"] = model_data["amount_value"].abs()
model_data["is_negative_amount"] = (model_data["amount_value"] < 0).astype(int)
model_data["hour"] = model_data["date"].dt.hour
model_data["day_of_week"] = model_data["date"].dt.dayofweek
model_data["month"] = model_data["date"].dt.month
model_data["has_error"] = model_data["errors"].notna().astype(int)

feature_columns = [
    "abs_amount",
    "is_negative_amount",
    "hour",
    "day_of_week",
    "month",
    "use_chip",
    "merchant_state",
    "mcc",
    "has_error",
]

X = model_data[feature_columns]
y = model_data["is_fraud"].astype(int)

pd.DataFrame({"feature": feature_columns})

## Optional Stratified Sampling

In [ ]:
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(X):
    sample_fraction = SAMPLE_SIZE / len(X)
    sample_index = (
        pd.DataFrame({"is_fraud": y})
        .groupby("is_fraud", group_keys=False)
        .sample(frac=sample_fraction, random_state=RANDOM_STATE)
        .index
    )
    X_model = X.loc[sample_index]
    y_model = y.loc[sample_index]
else:
    X_model = X
    y_model = y

sample_summary = pd.DataFrame(
    {
        "label": ["Non-fraud", "Fraud"],
        "rows": [int((y_model == 0).sum()), int((y_model == 1).sum())],
        "percent": [round((y_model == 0).mean() * 100, 4), round((y_model == 1).mean() * 100, 4)],
    }
)
sample_summary

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.25,
    stratify=y_model,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(X_train), len(X_test)],
        "fraud rows": [int(y_train.sum()), int(y_test.sum())],
        "fraud rate (%)": [round(y_train.mean() * 100, 4), round(y_test.mean() * 100, 4)],
    }
)
split_summary

## Build K-Means Pipeline

In [ ]:
numeric_features = ["abs_amount", "is_negative_amount", "hour", "day_of_week", "month", "has_error"]
categorical_features = ["use_chip", "merchant_state", "mcc"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

kmeans_model = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    batch_size=8192,
    n_init=10,
    random_state=RANDOM_STATE,
)

kmeans_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", kmeans_model),
    ]
)
print("K-Means pipeline created.")

## Train Model

In [ ]:
start_time = time.perf_counter()
kmeans_pipeline.fit(X_train)
print("K-Means model trained.")
training_time_seconds = time.perf_counter() - start_time
round(training_time_seconds, 2)

## Complexity and Efficiency Notes


In [ ]:
encoded_feature_count = len(kmeans_pipeline.named_steps["preprocess"].get_feature_names_out())
complexity_summary = pd.DataFrame(
    {
        "item": [
            "training rows",
            "test rows",
            "input features before encoding",
            "features after one-hot encoding",
            "number of clusters",
            "batch size",
            "training time seconds",
        ],
        "value": [
            len(X_train),
            len(X_test),
            len(feature_columns),
            encoded_feature_count,
            kmeans_pipeline.named_steps["model"].n_clusters,
            kmeans_pipeline.named_steps["model"].batch_size,
            round(training_time_seconds, 2),
        ],
    }
)
complexity_summary

## Cluster Evaluation Against Fraud Labels


In [ ]:
train_clusters = kmeans_pipeline.predict(X_train)
test_clusters = kmeans_pipeline.predict(X_test)

cluster_profile = (
    pd.DataFrame({"cluster": train_clusters, "is_fraud": y_train.to_numpy()})
    .groupby("cluster")
    .agg(rows=("is_fraud", "size"), fraud_count=("is_fraud", "sum"), fraud_rate=("is_fraud", "mean"))
    .reset_index()
)
cluster_profile["fraud_rate_percent"] = cluster_profile["fraud_rate"] * 100
risk_cluster = cluster_profile.sort_values("fraud_rate", ascending=False).iloc[0]["cluster"]
cluster_profile

In [ ]:
y_pred = (test_clusters == risk_cluster).astype(int)

distance_to_centers = kmeans_pipeline.named_steps["model"].transform(
    kmeans_pipeline.named_steps["preprocess"].transform(X_test)
)
anomaly_score = distance_to_centers.min(axis=1)

metrics_summary = pd.DataFrame(
    {
        "model": ["K-Means Clustering"],
        "accuracy": [accuracy_score(y_test, y_pred)],
        "roc_auc_distance_score": [roc_auc_score(y_test, anomaly_score)],
        "pr_auc_distance_score": [average_precision_score(y_test, anomaly_score)],
        "training_time_seconds": [training_time_seconds],
    }
)
metrics_summary.round(4)

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Non-fraud", "Fraud-risk cluster"], digits=4))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Non-fraud", "Fraud-risk cluster"],
    cmap="Purples",
    values_format=",d",
    ax=ax,
)
ax.set_title("K-Means Fraud-Risk Cluster Confusion Matrix")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "model_kmeans_confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()

## Cluster Fraud Rate Figure


## K-Means Cluster Profile


In [ ]:
cluster_details = X_train.copy()
cluster_details["cluster"] = train_clusters
cluster_details["is_fraud"] = y_train.to_numpy()

cluster_profile_extra = (
    cluster_details.groupby("cluster")
    .agg(
        transaction_count=("is_fraud", "size"),
        fraud_rate_percent=("is_fraud", lambda values: values.mean() * 100),
        average_amount=("abs_amount", "mean"),
        negative_amount_percent=("is_negative_amount", lambda values: values.mean() * 100),
        error_transaction_percent=("has_error", lambda values: values.mean() * 100),
    )
    .reset_index()
)

profile_plot = cluster_profile_extra.melt(
    id_vars="cluster",
    value_vars=["fraud_rate_percent", "negative_amount_percent", "error_transaction_percent"],
    var_name="cluster_metric",
    value_name="percent",
)

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(data=profile_plot, x="cluster_metric", y="percent", hue="cluster", ax=ax)
ax.set_title("K-Means Cluster Profile by Fraud, Negative Amount and Error Rate")
ax.set_xlabel("Cluster metric")
ax.set_ylabel("Percent (%)")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "model_kmeans_cluster_profile.png", dpi=160, bbox_inches="tight")
plt.show()

cluster_profile_extra.round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=cluster_profile, x="cluster", y="fraud_rate_percent", hue="cluster", legend=False, ax=ax)
ax.set_title("Fraud Rate by K-Means Cluster")
ax.set_xlabel("Cluster")
ax.set_ylabel("Fraud rate (%)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "model_kmeans_cluster_fraud_rate.png", dpi=160, bbox_inches="tight")
plt.show()

## Precision-Recall Curve Using Distance Score



In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, anomaly_score)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(recall, precision, color="#7e22ce")
ax.set_title("K-Means Precision-Recall Curve using Distance Score")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "model_kmeans_precision_recall_curve.png", dpi=160, bbox_inches="tight")
plt.show()

## Silhouette Score Sample


In [ ]:
silhouette_sample_size = min(10_000, len(X_test))
silhouette_sample = X_test.sample(n=silhouette_sample_size, random_state=RANDOM_STATE)
silhouette_features = kmeans_pipeline.named_steps["preprocess"].transform(silhouette_sample)
silhouette_labels = kmeans_pipeline.named_steps["model"].predict(silhouette_features)
silhouette = silhouette_score(silhouette_features, silhouette_labels)
round(silhouette, 4)

## Save Model



In [ ]:
model_path = MODEL_DIR / "kmeans_fraud_cluster_model.joblib"
joblib.dump(kmeans_pipeline, model_path)
model_path